# Q1A | Summarize the Data

> *Summarize the data. How much data is present? What attributes/features are continuous valued? Which attributes are categorical?*

This notebook answers the question in three parts:

1. **How much data is present?** Row and column counts, the train/test split, duplicates, and where values are missing.
2. **Which attributes are continuous?** Data types, the number of distinct values, and the range of each attribute.
3. **Which attributes are categorical?** The conclusion that follows from the evidence in part 2.

The **Answer Summary** below pulls the three parts together.

---

#### **IMPORTANT**

This summary combines the `train` and `test` datasets to describe the full dataset. It only counts and describes the data. No modeling decisions are made here, so looking at the test data does not introduce `data snooping` bias.

# Answer Summary

- **How much data is present?** 1,893 rows and 9 columns (1 target, `Life Ladder`, and 8 features), split 82.36% train (1,559 rows) and 17.64% test (334 rows). There are no duplicate rows. 283 values are missing, all in the train set, across 185 rows (11.87% of train rows). `Perceptions of corruption` (86) and `Generosity` (71) have the most missing values, and the target has none.
- **Which attributes are continuous?** All 9. Every column holds numeric values with 372 to 1,519 distinct values spread across a range, and no value repeats more than 17 times.
- **Which attributes are categorical?** None. No column holds labels or a small set of repeated codes.

---

# Imports

In [1]:
import pandas as pd

from Q1.consts import *

from utils import read_csv_to_df

# Fetch Data

In [2]:
train_df = read_csv_to_df(path=TRAIN_PATH)
test_df = read_csv_to_df(path=TEST_PATH)
combined_df = pd.concat([train_df, test_df], ignore_index=True)

# 1. How Much Data is Present?

In [3]:
summary = pd.DataFrame(
	{
		"row count": [train_df.shape[0], test_df.shape[0], combined_df.shape[0]],
		"% row count": [
			round(train_df.shape[0] / combined_df.shape[0] * 100, 2),
			round(test_df.shape[0] / combined_df.shape[0] * 100, 2),
			round(combined_df.shape[0] / combined_df.shape[0] * 100, 2),
		],
		"columns count": [train_df.shape[1], test_df.shape[1], combined_df.shape[1]],
		"missing values": [train_df.isnull().sum().sum(), test_df.isnull().sum().sum(), combined_df.isnull().sum().sum()],
		"rows w/ missing values": [
			train_df.isnull().any(axis=1).sum(),
			test_df.isnull().any(axis=1).sum(),
			combined_df.isnull().any(axis=1).sum()
		],
		"% rows w/ missing values": [
			round(train_df.isnull().any(axis=1).sum() / train_df.shape[0] * 100, 2),
			round(test_df.isnull().any(axis=1).sum() / test_df.shape[0] * 100, 2),
			round(combined_df.isnull().any(axis=1).sum() / combined_df.shape[0] * 100, 2),
		]
	},
	index=["train_df", "test_df", "combined_df"],
)
s = summary.T
display(s)

combined_df.info()

,train_df,test_df,combined_df
row count,1559.00,334.00,1893.00
% row count,82.36,17.64,100.00
columns count,9.00,9.00,9.00
missing values,283.00,0.00,283.00
rows w/ missing values,185.00,0.00,185.00
% rows w/ missing values,11.87,0.00,9.77


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1893 entries, 0 to 1892
Data columns (total 9 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Life Ladder                       1893 non-null   float64
 1   Log GDP per capita                1867 non-null   float64
 2   Social support                    1883 non-null   float64
 3   Healthy life expectancy at birth  1856 non-null   float64
 4   Freedom to make life choices      1871 non-null   float64
 5   Generosity                        1822 non-null   float64
 6   Perceptions of corruption         1807 non-null   float64
 7   Positive affect                   1876 non-null   float64
 8   Negative affect                   1879 non-null   float64
dtypes: float64(9)
memory usage: 133.2 KB


- There are 1,893 rows and 9 columns (1 target, `Life Ladder`, and 8 features), split 82.36% train (1,559 rows) and 17.64% test (334 rows).
- There are 283 missing values in the training data affecting 11.87% of the training rows and no missing values in the testing data.
- All values are `float64` types.
- There are no "identifier" columns (like country name, year, row ID) so each row is an anonymous observation.
- 283 missing values spread over 185 rows means some rows are missing more than one value.

## Duplicate Rows

In [4]:
duplicates = pd.DataFrame(
	{
		"duplicate rows": [
			train_df.duplicated().sum(),
			test_df.duplicated().sum(),
			combined_df.duplicated().sum(),
		],
	},
	index=["train_df", "test_df", "combined_df"],
)

duplicates

,duplicate rows
train_df,0
test_df,0
combined_df,0


- There are no duplicated rows within either dataset, and none in the combined data, so no row appears in both train and test.
- Every row is a distinct observation, so the test set does not contain a copy of any training row.

## Missing Values by Feature

In [5]:
missing_by_feature = pd.DataFrame({
	'Missing (train)': train_df.isna().sum(),
	'% of rows (train)': (train_df.isna().sum() / len(train_df) * 100).round(2),
	'Missing (test)': test_df.isna().sum(),
	'% of rows (test)': (test_df.isna().sum() / len(test_df) * 100).round(2),
}).sort_values('Missing (train)', ascending=False)

missing_by_feature

,Missing (train),% of rows (train),Missing (test),% of rows (test)
Perceptions of corruption,86,5.52,0,0.0
Generosity,71,4.55,0,0.0
Healthy life expectancy at birth,37,2.37,0,0.0
Log GDP per capita,26,1.67,0,0.0
Freedom to make life choices,22,1.41,0,0.0
Positive affect,17,1.09,0,0.0
Negative affect,14,0.90,0,0.0
Social support,10,0.64,0,0.0
Life Ladder,0,0.00,0,0.0


- `Perceptions of corruption` (5.52%) and `Generosity` (4.55%) have the most gaps and account for 157 (55.5%) of the 283 missing training values.
- Every feature is missing fewer than 6% of its values, so no feature is too sparse to use.
- The target, `Life Ladder`, has no missing values, so no row has to be excluded for a missing target.

## Missing Values per Row

In [6]:
missing_per_row = train_df.isna().sum(axis=1).value_counts().sort_index()

missing_per_row_df = pd.DataFrame({
	"train rows": missing_per_row,
	"% of train rows": (missing_per_row / len(train_df) * 100).round(2),
})
missing_per_row_df.index.name = "missing values in row"

missing_per_row_df

,train rows,% of train rows
missing values in row,,
0,1374,88.13
1,130,8.34
2,27,1.73
3,23,1.48
6,5,0.32


- Most incomplete rows are missing only 1 value.
- 5 rows are missing 6 of the 8 features (75%). How to handle them is a modeling decision, so it is not made here.

# 2. Which Attributes are Continuous?

An attribute is continuous if it measures a quantity that can take any value within a range. That depends on what the values mean, not on how they are stored, so two checks are used as evidence:

1. **Are the values numeric?** A continuous attribute must hold numbers, but the stored data type alone is not proof either way. Numbers can be stored as text and categories can be stored as numbers.
2. **Do the values behave like measurements?** Many distinct values, fractional values, and a spread across a range point to a measured quantity. A handful of repeated values points to codes or categories.

In [13]:
info_df = pd.DataFrame({
	"dtype": combined_df.dtypes,
	"non-null values": combined_df.count(),
	"unique values": combined_df.nunique(),
	"most frequent value count": combined_df.apply(lambda col: col.value_counts().iloc[0]),
}).sort_values(by=["unique values", "most frequent value count"], ascending=False)

info_df

,dtype,non-null values,unique values,most frequent value count
Life Ladder,float64,1893,1519,5
Log GDP per capita,float64,1867,1476,4
Healthy life expectancy at birth,float64,1856,825,15
Generosity,float64,1822,602,13
Perceptions of corruption,float64,1807,566,14
Freedom to make life choices,float64,1871,533,10
Social support,float64,1883,452,15
Positive affect,float64,1876,430,12
Negative affect,float64,1879,372,17


- Every column is `float64`, so every value is a number. Whether those numbers are measurements or codes is settled by the distinct value counts.
    - Every column has hundreds of distinct values, far more than a set of category labels.
- No single value dominates. The most common value in any column repeats at most 17 times out of about 1,800 rows.

## Value Ranges

In [16]:
value_ranges = pd.DataFrame({
	"min": combined_df.min(),
	"max": combined_df.max(),
	"mean": combined_df.mean().round(3),
	"std": combined_df.std().round(3),
    "whole numbers only": combined_df.apply(lambda col: (col.dropna() % 1 == 0).all()),
}, index=combined_df.columns)

value_ranges

,min,max,mean,std,whole numbers only
Life Ladder,2.375,8.019,5.461,1.124,False
Log GDP per capita,6.635,11.648,9.356,1.156,False
Social support,0.290,0.987,0.812,0.119,False
Healthy life expectancy at birth,32.300,77.100,63.337,7.546,False
Freedom to make life choices,0.258,0.985,0.741,0.142,False
Generosity,-0.335,0.698,-0.000,0.162,False
Perceptions of corruption,0.035,0.983,0.748,0.187,False
Positive affect,0.322,0.944,0.709,0.107,False
Negative affect,0.083,0.705,0.269,0.085,False


- Every attribute takes values across a **range**, not from a fixed list.
- The values are fractional. Every minimum and maximum in the table has a decimal part.
- The column names describe measured quantities (ex: `Healthy life expectancy at birth`, `Log GDP per capita`), not labels or groups.
- Five features (`Social support`, `Freedom to make life choices`, `Perceptions of corruption`, `Positive affect`, `Negative affect`) fall between 0 and 1.
- `Healthy life expectancy at birth` ranges from 32.3 to 77.1.
- `Generosity` has a mean of about 0 and is the only attribute with negative values (min -0.335).

- The features use very different ranges (0 to 1 vs. 32.3 to 77.1), which is worth noting for any model that is sensitive to feature scale.

**All 9 attributes (the target and all 8 features) are continuous.**

# 3. Which Attributes are Categorical?

**No attributes are categorical.** Section 2 showed that all 9 columns hold numeric values with hundreds of distinct values each (372 to 1,519), so none of them is a set of labels or codes.